# Create and Review Metadata Files

This notebook prepares `metadata.json`. It never writes while cells are merely executed. First inspect the exact before/after comparison, then choose **Confirm overwrite** or **Reject changes**.

`metadata.json` describes the original measurement and its preprocessing. New Module 13 reuse metadata is created later by Lab 13 as `metadata_reused.json`.

<div style="border: 3px solid #b45309; background: #fff7ed; padding: 18px 20px; margin: 16px 0 24px 0; border-radius: 8px;">
<h2 style="margin-top: 0; color: #9a3412;">Important: Metadata are not normally generated all at once</h2>
<p>In a real research workflow, metadata come from different sources. Some metadata are <strong>recorded automatically</strong> by instruments or software, some are <strong>extracted from the recorded files</strong>, and some must be <strong>entered or reviewed manually</strong> by the researchers who understand the experiment.</p>
<p>In the JupyterHub environment used for this course, JSON files cannot be edited manually. This notebook therefore provides a structured way to review, supplement, and write the metadata. It does not mean that all metadata should normally be invented or generated automatically.</p>
<p><strong>Your task is still to check where each value comes from and whether it correctly describes your measurement.</strong></p>
</div>

## Section 0: Choose the Preparation Mode

`metadata.json` is the central description of your measurement: it points to the recorded data file, names the use case (`measurement_type`) and measured `quantity`, and stores the analysis parameters that Lab 6 reads and the RO-Crate export packages. It holds the configuration for **both** use cases (drivetrain and suspension) side by side; the top-level fields select which one is currently active.

This notebook never edits the file directly. Instead, it builds a **draft**: the complete possible new version of `metadata.json`, assembled in memory from your inputs. You inspect and validate the draft first; only a button click in Section 5 writes it to disk.

The mode decides how the draft is built:

| Mode | What happens to `metadata.json` |
|---|---|
| `'load'` | The file is only **read and validated**. The inputs in Sections 2 and 3 are ignored; the validation simply checks the file as it is. No write buttons are shown - nothing can change the file in this mode. |
| `'replace'` | A completely **new draft** is built from the central defaults in `metadata_loader.py`. Your explicit inputs in Sections 2 and 3 override those defaults. On confirmation, the whole file is overwritten - including the *other* use case, which is reset to its defaults. |
| `'update'` | The draft **starts from the current file contents**. Only values you explicitly set in Sections 2 and 3 are changed; everything else - especially the unselected use case - is preserved exactly as it is. |

In modes `'replace'` and `'update'`, running cells only prepares and previews the draft. The file on disk is written **only** when you click **Confirm overwrite** in Section 5, after seeing the exact before/after comparison in Section 4.

In [ ]:
# Section 0: Metadata preparation mode
# ============================== YOUR INPUT ==================================
metadata_mode = 'load'   # 'load' | 'replace' | 'update' - see the table above
# ============================================================================

valid_modes = {
    'load': 'Load and validate only',
    'replace': 'Prepare complete replacement',
    'update': 'Prepare update of selected use case',
}
if metadata_mode not in valid_modes:
    raise ValueError(f"metadata_mode must be one of {sorted(valid_modes)}")
print(valid_modes[metadata_mode])

## Section 1: Load Helpers and Existing File

The validation and write logic lives in `src/`; the notebook only supplies values and displays the result.

In [ ]:
# Section 1: Imports and current file contents
from pathlib import Path
import sys

from IPython.display import HTML, display

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import (
    KEEP,
    load_json_file,
    load_public_metadata,
    public_metadata_path,
)
from metadata_validation import (
    apply_use_case_overrides,
    create_metadata_write_controls,
    display_draft_validation_and_preview,
    prepare_metadata_draft,
    resolve_general_metadata_inputs,
    validate_public_metadata,
)

metadata_file = public_metadata_path(project_root)
public_before_raw = load_json_file(metadata_file, {})
existing_public_metadata = load_public_metadata(project_root)

if metadata_file.is_file():
    print(f'Loaded existing {metadata_file.name}. No file has been changed.')
else:
    print(f'WARNING: {metadata_file.name} does not exist yet.')
    print("Central defaults are shown instead. Use mode 'replace' to create the file.")

## Section 2: Dataset and General Metadata

The first cell is your **input cell**: every value starts as `KEEP`, which means "keep the value from the base metadata" (the existing file in mode `'update'`, the central defaults in mode `'replace'`). Replace `KEEP` only where you want to set a value yourself. In mode `'load'` these inputs are ignored. `measurement_type` defines the use case; `quantity` is derived from it automatically.

The second cell processes your inputs and shows for each value whether it was set here or kept from the base.

In [ ]:
# Section 2: Edit general public metadata
# ============================== YOUR INPUT ==================================
# KEEP means: keep the value from the base metadata.
# Replace KEEP only where you want to set a value yourself.
recorded_data_path = KEEP   # path to the recorded measurement file, e.g. 'data/suspension/Example/Beschleunigung ohne g.xls'
measurement_type = KEEP     # use case: 'drivetrain' or 'suspension'
run_name = KEEP             # short name of this measurement run, used in file and folder names, e.g. 'Example' or 'group_c_run_2'
quantity = KEEP             # measured quantity; derived automatically from measurement_type
data_stage = KEEP           # processing stage of the data, e.g. 'raw'
version = KEEP              # dataset version, e.g. 'v0.1.0'
hot_storage_path = 'output' # folder for files created while actively working with the data ("hot" storage, as opposed to archived "cold" storage)
# ============================================================================

In [ ]:
# Section 2: Combine your inputs with the base metadata
base_metadata, general_inputs, effective_general = resolve_general_metadata_inputs(
    metadata_mode,
    existing_public_metadata,
    recorded_data_path=recorded_data_path,
    measurement_type=measurement_type,
    run_name=run_name,
    quantity=quantity,
    data_stage=data_stage,
    version=version,
    hot_storage_path=hot_storage_path,
)

## Section 3: Selected Use-Case Parameters

Only the selected `measurement_type` is relevant here. The first cell is your **input cell**: enter **only the keys you want to change** into the two override dictionaries; every other value keeps the base value (existing file in mode `'update'`, central defaults in mode `'replace'`). The second cell applies the overrides and prints the resulting parameters so you can check the effect before building the preview in Section 4.

In [ ]:
# Section 3: Selected use-case parameters
# ============================== YOUR INPUT ==================================
# Enter only the keys you want to change; everything else keeps the base
# value. Nested dictionaries are merged key by key.
analysis_overrides = {
    # example: 'smoothing_window': 15,
    # example: 'outlier_z_threshold': 2.5,
}
setup_overrides = {
    # example (suspension): 'acceleration_unit': 'm/s^2',
    # example (drivetrain): 'rotor_marker': {'bright_dark_cycles_per_rotation': 2},
}
# ============================================================================

In [ ]:
# Section 3: Apply the overrides to the selected use case
edited_analysis_metadata, edited_setup_metadata = apply_use_case_overrides(
    base_metadata,
    effective_general,
    analysis_overrides,
    setup_overrides,
)

## Section 4: Validate and Preview Before/After

Run this cell after editing Sections 2 and 3. It builds the **draft** - the complete possible new version of `metadata.json` - validates it, and shows exactly what would change. Validation errors disable the confirmation button; warnings remain confirmable. In modes `'replace'` and `'update'`, every changed path is shown with its previous and proposed value. In mode `'load'`, the current file contents are shown instead, because nothing will be written.

In [ ]:
# Section 4: Build the draft, validate, and show exact differences
metadata_draft = prepare_metadata_draft(
    existing_public_metadata,
    metadata_mode,
    general_inputs,
    edited_analysis_metadata,
    edited_setup_metadata,
)
draft_validation = display_draft_validation_and_preview(
    metadata_mode,
    public_before_raw,
    metadata_draft,
    project_root,
)

## Section 5: Confirm or Reject

The red button writes the draft from Section 4 to `metadata.json`. The green button explicitly rejects the draft. If this notebook is run with **Run All**, execution continues without writing while the decision remains pending.

**After changing any value in Section 2 or 3, use "Restart" and "Run All" before confirming.** The buttons write exactly the draft previewed in Section 4; running cells out of order can otherwise write an outdated draft. Run All is safe here - nothing is written without a button click.

In [ ]:
# Section 5: Explicit write decision
if 'metadata_write_widget' in globals():
    metadata_write_widget.close()

if metadata_mode == 'load':
    metadata_write_state = {'decision': 'load_only'}
    display(HTML("<div style='color:#1f7a3a;font-weight:600'>Mode 'load': metadata.json will not be written.</div>"))
else:
    metadata_write_widget, metadata_write_state = create_metadata_write_controls(
        project_root,
        public_before_raw,
        metadata_draft,
    )
    display(metadata_write_widget)

## Section 6: Verify the Decision

After clicking a button, run this cell again. It clearly distinguishes written, rejected, pending, and load-only states.

In [ ]:
# Section 6: Verify current on-disk state
decision = metadata_write_state.get('decision')
if decision in ('written', 'load_only'):
    verified_metadata = load_public_metadata(project_root)
    verification = validate_public_metadata(verified_metadata, project_root)
    print('Decision:', decision)
    print('metadata.json valid:', verification['valid'])
    print('Selected data exists:', (project_root / verified_metadata['recorded_data_path']).is_file())
elif decision == 'rejected':
    print('Changes were rejected. No file was written.')
elif decision == 'pending':
    print('Waiting for your button decision. No file has been written.')
else:
    print('Decision state:', decision, '- rebuild the preview before writing.')